In [ ]:
"""
CommentsPostProcessingGPT5.ipynb.

------------------
This is the large-language model-based text analysis notebook
------------------

It performs the following:
  * Topic Summarization
  * Topic Classification
  * Phrase Extraction
    * Image Topic Extraction
    * VisTypes Topic Analysis
  * Keyword Extraction
  * Keyword Count
  * Keywords Curating
"""

In [ ]:
# --- portable setup (added by clean-up) ---
import os, sys
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
os.makedirs('data_processing', exist_ok=True)
from data_loader import load_sheet, load_paired_comparison
import config

# Set your key in the environment before running:  os.environ['OPENAI_API_KEY'] = '...'


In [ ]:
import pandas as pd
from openai import OpenAI
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import defaultdict, Counter

# read api_key and init gpt client
os.environ.setdefault("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [ ]:
VisTypes = ['Area','Bar','Cont.-ColorPatn','Glyph','Grid','Line','Node-link','Point','Text']

topic_names = [
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
    'Immediacy / Cognitive Load'
]

# 1. Topic Summarization

In [ ]:
def summarize_visual_complexity(file_path="comments.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        comments = f.read()

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert in visualization and linguistics. "
                    "Analyze participant responses about why one visualization "
                    "seems more visually complex than another.\n\n"
                    "Your task is to:\n"
                    "- identify the main factors participants mention\n"
                    "- group similar explanations into less than 10 coherent themes\n"
                    "- assign a clear, meaningful, and concise topic label to each theme\n"
                    "- explain WHY that theme contributes to perceived visual complexity\n"
                    "- base the themes ONLY on what appears in the comments\n\n"
                    "OUTPUT FORMAT (strict):\n"
                    "Numbered list\n"
                    "Each item must follow:\n"
                    "1. **Topic Name**: Explanation in 2–3 sentences.\n\n"
                    "Do NOT:\n"
                    "- invent topics that do not appear in the comments\n"
                    "- include introductions or conclusions\n"
                    "- include bullet points\n"
                    "- include citations or references\n"
                )
            },
            {
                "role": "user",
                "content": (
                    "Here are the participant responses (one per line):\n\n"
                    f"{comments}\n\n"
                    "Now produce the structured summary."
                )
            }
        ],
    )

    return response.choices[0].message.content.strip()


summary = summarize_visual_complexity(file_path="comments.txt")
print(summary)

The above topics have been reviewed and revised to the ones listed in the prompt of the Topic Classification task below

# 2. Topic Classification

In [ ]:
spreadsheet_id = "1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc"
sheet_id = "1103808983"
df_original = load_sheet(spreadsheet_id, sheet_id)
print(f'Number of records is {len(df_original)}')

df = df_original
df = df[df['ToRemove']!=1]
print(f'Number of records after filtering is {len(df)}')
df.columns

In [ ]:
# count number of unique images
less_cmplx = (
    df['lessComplexImageName']
    .astype(str)
    .str.strip()
    .replace({'nan': None, '': None})
    .dropna()
)

more_cmplx = (
    df['moreComplexImageName']
    .astype(str)
    .str.strip()
    .replace({'nan': None, '': None})
    .dropna()
)

all_imgs = set(less_cmplx).union(set(more_cmplx))
len(all_imgs)

In [ ]:
def gpt_classify(text):
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "user",
                "content": f"""
Participants compared pairs of visualization images and explained why one seemed more visually complex. Classify the participant's response into at least one of the topics below, don't invent new topics. It is guaranteed that the response will match at least one of the topics.

Topics:
1. Density / Clutter: The degree to which observers can discriminate newly added or salient items in a scene without interference from surrounding elements.
2. Encoding Clarity: The degree to which observers can clearly separate visual elements from the data they represent, and understand the mapping between them.
3. Semantics / Text Legibility: The degree to which observers can recognize labels, legends, axes, and textual content, and use them to interpret the meaning of visual marks.
4. Schema: The degree to which observers immediately perceive familiarity or recognizability in the visual form—such as common textures, depths, or representational types.
5. Color, Symbol, and Texture Details: The degree to which observers understand fine visual details, 3D depth cues, magnification regions, and symbol compositions.
6. Immediacy / Cognitive Load: The effort required to interpret the visualization, including the time needed for inspection, comparison, and mental processing.
7. Aesthetics Uncertainty: The degree of uncertainty observers experience when trying to clearly perceive structure, meaning, or color harmony in the visualization.

Participant response:
{text}

Output format:
A list of relevant topics, separated by "; " (semicolon and space). No additional text.

Example output:
Color Complexity; Element Density / Visual Clutter
""",
            },
        ],
    )

    try:
        content = response.choices[0].message.content
        return content
    except:
        return "--"


# Example usage
gpt_classify("""Data is easier to compare on the left than on the right.""")

In [ ]:
def batch_classify(idx, row):
    comment = row.commentCollected
    if pd.isna(comment):
        return idx, None
    categories = gpt_classify(comment)
    return idx, categories

results = []
with ThreadPoolExecutor() as executor:
    futures = {
        executor.submit(batch_classify, idx, row): idx for idx, row in df.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures)):
        idx, categories = future.result()
        if categories is not None:
            results.append((idx, categories))

for idx, categories in results:
    df.loc[idx, "ExtractedTopics"] = categories

df[['index', 'ExtractedTopics']].to_csv('data_processing/extracted_topics.csv', index=False)
df.head()

# 3. Phrase Extraction

In [ ]:
topic_def = {
    "Density / Clutter": "Density / Clutter: The degree to which observers can discriminate newly added or salient items in a scene without interference from surrounding elements.",
    "Encoding Clarity": "Encoding Clarity: The degree to which observers can clearly separate visual elements from the data they represent, and understand the mapping between them.",
    "Semantics / Text Legibility": "Semantics / Text Legibility: The degree to which observers can recognize labels, legends, axes, and textual content, and use them to interpret the meaning of visual marks.",
    "Schema": "Schema: The degree to which observers immediately perceive familiarity or recognizability in the visual form—such as common textures, depths, or representational types.",
    "Color, Symbol, and Texture Details": "Color, Symbol, and Texture Details: The degree to which observers understand fine visual details, 3D depth cues, magnification regions, and symbol compositions.",
    "Immediacy / Cognitive Load": "Immediacy / Cognitive Load: The effort required to interpret the visualization, including the time needed for inspection, comparison, and mental processing.",
    "Aesthetics Uncertainty": "Aesthetics Uncertainty: The degree of uncertainty observers experience when trying to clearly perceive structure, meaning, or color harmony in the visualization."
}

def phrase_extract_prompt(topics, comment):
    topics_str = ""
    for topic in topics.split("; "):
        topic = topic.strip()
        topics_str += topic_def[topic] + "\n"
    topics_str = topics_str.strip()
    return f'''
Participants compared pairs of visualization images and explained why one seemed more visually complex.
We have identified that the topics listed below were mentioned in the participant’s response.
For each topic, extract the exact words/phrases from the participant’s comment that directly relate to that topic.
There will be at least one match for each topic.
Do not modify, paraphrase, or summarize them—use the exact wording as it appears in the response.
If a short match is part of a long match, keep the long match only.

The same word/phrase can be associated with multiple topics if applicable. Use of external tools or programming is prohibited.

Your output should be a valid JSON string formatted as follows:
{{ "topic1": ["<word/phrase>", "<word/phrase>"], "topic2": ["<word/phrase>"], ... }}
Do not wrap the JSON in code blocks, and do not include any extra escape characters or newline characters that would interfere with json.loads() parsing.

Participant’s response:
"""
{comment}
"""

Identified topics and explainations:
"""
{topics_str}
"""
'''

def gpt_phrase_extract(text):

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "user",
                "content": text,
            },
        ],
    )

    try:
        content = response.choices[0].message.content
        return json.dumps(json.loads(content), indent=2)
    except:
        return "--NOT JSON--\n" + content

In [ ]:
def batch_phrase_extract(idx, row):
    try:
        comment = row.commentCollected
        if pd.isna(comment):
            return idx, None
        topics = row.ExtractedTopics
        prompt = phrase_extract_prompt(topics, comment)
        extracted = gpt_phrase_extract(prompt)
        return idx, extracted
    except Exception as e:
        print(e)
        return idx, None

results = []
with ThreadPoolExecutor() as executor:
    futures = {
        executor.submit(batch_phrase_extract, idx, row): idx
        for idx, row in df.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            idx, extracted = future.result()
            if extracted is not None:
                results.append((idx, extracted))
        except Exception as e:
            print(e)


for idx, extracted in results:
    df.loc[idx, "ExtractedPhrases"] = extracted

df[['index', 'ExtractedPhrases']].to_csv('data_processing/extracted_phrases.csv', index=False)
df.head()

## 3.1 Batch extract phrases into more-complex or less-complex column with sentiment

In [ ]:
def phrase_extract_sentiment_prompt(topics, comment):
    topics_str = ""
    for topic in topics.split("; "):
        topic = topic.strip()
        topics_str += topic_def[topic] + "\n"
    topics_str = topics_str.strip()

    if pd.isna(comment):
        comment=''

    return f'''
Participants compared pairs of visualization images and explained why one seemed more visually complex or simple.
The participant’s response on one of the images is provided below, and may or may not be related to the topics listed below.
If the participant’s response is related to one topic, extract the exact words/phrases from the participant’s comment that directly relate to this topic.
Do not modify, paraphrase, or summarize them—use the exact wording as it appears in the response.
If a short match is part of a long match, keep the long match only.

The same word/phrase can be associated with multiple topics if applicable. Use of external tools or programming is prohibited.

In addition, for a matched word/phrase, determine if it expresses increased complexity (+) or decreased complexity (-), and add (+) or (-) to the front of the match.

Your output should be a valid JSON string formatted as follows:
{{ "topic1": ["(+) word/phrase", "(-) word/phrase"], "topic2": ["(+) word/phrase"], ... }}
If no match is found at all, return an empty JSON object.
Do not wrap the JSON in code blocks, and do not include any extra escape characters or newline characters that would interfere with json.loads() parsing.

Participant’s response:
"""
{comment}
"""

Identified topics and explainations:
"""
{topics_str}
"""
'''

In [ ]:
def batch_phrase_extract_select(idx, row, select):
    try:
        comment = row[select]  # CommentCollectedOnMoreComplexFig or CommentCollectedOnLessComplexFig
        if pd.isna(comment):
            return idx, None
        topics = row.ExtractedTopics
        prompt = phrase_extract_sentiment_prompt(topics, comment)
        extracted = gpt_phrase_extract(prompt)
        return idx, extracted
    except Exception as e:
        print(e)
        return idx, None


# set which comment to use
select = "CommentCollectedOnMoreComplexFig" # CommentCollectedOnMoreComplexFig or CommentCollectedOnLessComplexFig

results = []
with ThreadPoolExecutor() as executor:
    futures = {
        executor.submit(batch_phrase_extract_select, idx, row, select): idx
        for idx, row in df.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            idx, extracted = future.result()
            if extracted:
                results.append((idx, extracted))
            else:
                results.append((idx, ''))
        except Exception as e:
            print(e)

# save results
if select == "CommentCollectedOnMoreComplexFig":
    result_col = "ExtractedPhrasesMoreComplex"
elif select == "CommentCollectedOnLessComplexFig":
    result_col = "ExtractedPhrasesLessComplex"
for idx, extracted in results:
    df.loc[idx, result_col] = extracted

df[['index', result_col]].to_csv(f'data_processing/{result_col}.csv', index=False)
df[['index', select, result_col]].head()

## 3.2 Count Topics for Each Image

In [ ]:
# open 2025EuroVis main table to get Vis_Types
spreadsheet_id = "1eJ3VUh2ZKZy6S6qo9nzF8h0rwwMNBhwuK42wdgeS54w"
sheet_id = "1073958835"
vis_map = load_sheet(spreadsheet_id, sheet_id)
# vis_map = vis_map[['ImageName', 'VisTypes']]
vis_map = vis_map.set_index("ImageName")
print(f'Number of images is {len(vis_map)}')
vis_map.head()

In [ ]:
# extract image topics
image_topics = {}

more_with_topics = set()
less_with_topics = set()

more_all_imgs = set()
less_all_imgs = set()

for idx, row in df.iterrows():

    # MORE COMPLEX
    img_more = row.get("moreComplexImageName", None)
    phrases_more = row.get("ExtractedPhrasesMoreComplex", "")

    if isinstance(img_more, str) and img_more.strip():
        img_more = img_more.strip()
        more_all_imgs.add(img_more)

    if isinstance(phrases_more, str) and phrases_more.strip():
        try:
            obj = json.loads(phrases_more)

            if obj:  # must contain topics
                for topic in obj.keys():
                    image_topics.setdefault(img_more, set()).add(topic)

                more_with_topics.add(img_more)
        except:
            pass

    # LESS COMPLEX
    img_less = row.get("lessComplexImageName", None)
    phrases_less = row.get("ExtractedPhrasesLessComplex", "")

    if isinstance(img_less, str) and img_less.strip():
        img_less = img_less.strip()
        less_all_imgs.add(img_less)

    if isinstance(phrases_less, str) and phrases_less.strip():
        try:
            obj = json.loads(phrases_less)

            if obj:  # must contain topics
                for topic in obj.keys():
                    image_topics.setdefault(img_less, set()).add(topic)

                less_with_topics.add(img_less)
        except:
            pass

# build final output table
rows = []
rows_not_found = []
for img, topics in image_topics.items():
    topics_str = "; ".join(sorted(topics))

    vis_type = vis_map.loc[img, "VisTypes"] if img in vis_map.index else ""
    if vis_type == "":
        rows_not_found.append(img)
    else:
        rows.append((img, vis_type, topics_str))

output_df = pd.DataFrame(rows, columns=["ImageName", "VisTypes", "Topics"])
output_df.to_csv("data_processing/image_topics_output.csv", index=False)

# compute missing
more_without_topics = more_all_imgs - more_with_topics
less_without_topics = less_all_imgs - less_with_topics

print(f"Images WITH topics (More-Complex): {len(more_with_topics)}")
print(f"Images WITHOUT topics (More-Complex): {len(more_without_topics)}")

print(f"Images WITH topics (Less-Complex): {len(less_with_topics)}")
print(f"Images WITHOUT topics (Less-Complex): {len(less_without_topics)}")

print(f"Unique total WITH topics: {len(more_with_topics.union(less_with_topics))}")
print(f"Unique total WITHOUT topics: {len(more_without_topics.union(less_without_topics))}")

output_df.head()

# 4. Keyword Extraction

In [ ]:
topic_def = {
    "Density / Clutter": "Density / Clutter: The degree to which observers can discriminate newly added or salient items in a scene without interference from surrounding elements.",
    "Encoding Clarity": "Encoding Clarity: The degree to which observers can clearly separate visual elements from the data they represent, and understand the mapping between them.",
    "Semantics / Text Legibility": "Semantics / Text Legibility: The degree to which observers can recognize labels, legends, axes, and textual content, and use them to interpret the meaning of visual marks.",
    "Schema": "Schema: The degree to which observers immediately perceive familiarity or recognizability in the visual form—such as common textures, depths, or representational types.",
    "Color, Symbol, and Texture Details": "Color, Symbol, and Texture Details: The degree to which observers understand fine visual details, 3D depth cues, magnification regions, and symbol compositions.",
    "Immediacy / Cognitive Load": "Immediacy / Cognitive Load: The effort required to interpret the visualization, including the time needed for inspection, comparison, and mental processing.",
    "Aesthetics Uncertainty": "Aesthetics Uncertainty: The degree of uncertainty observers experience when trying to clearly perceive structure, meaning, or color harmony in the visualization."
}


def keyword_extract_prompt(extracted):
    topics_str = ""
    for topic in json.loads(extracted).keys():
        # topic = topic.strip()
        topics_str += topic_def[topic] + "\n"
    topics_str = topics_str.strip()

    return f"""
Participants compared pairs of visualization images and explained why one seemed more or less visually complex.
You are given:
(1) the identified visualization complexity topics and their definitions, and
(2) the extracted phrases from the participant’s response for each topic.

Your task:
For each topic, extract meaningful **keywords** that are either:
  • (A) nouns or noun phrases (visual features), or
  • (B) adjectives or adjective phrases (perceptual, aesthetic, cognitive qualities).

===========================================================
KEYWORD EXTRACTION RULES (APPLY TO ALL 7 TOPICS)
===========================================================

1. Keywords must come directly from participant phrases.
   - Do NOT invent new terms.
   - You may singularize nouns or convert adjectives to base form.
   - You may keep multiword adjective/noun phrases when meaningful.

2. Extract ONLY two types of terms:

   A. **Nouns or noun phrases** describing visualization features:
      Examples:
        "line", "shape", "axis", "label", "text label",
        "color scale", "data point", "grid", "legend",
        "visual element", "overlap", "clutter", "pattern",
        "detail", "depth cue", "symbol", "annotation"

   B. **Adjectives or adjective phrases** describing perceptual or cognitive qualities:
      Examples:
        - density & clutter: "cluttered", "dense", "sparse", "busy"
        - semantic clarity: "ambiguous", "confusing", "clear", "readable"
        - detail & granularity: "fine", "detailed", "textured"
        - aesthetics/color: "appealing", "colorful", "bright", "dull"
        - cognitive load: "difficult", "overwhelming", "easy", "intuitive"
        - form/schema: "familiar", "unfamiliar", "structured", "irregular"
        - expressive/aesthetic: "expressive", "engaging", "elegant"

   These must represent **meaningful VIS-relevant concepts**.

3. Do NOT extract:
   - verbs (“look”, “show”, “represent”, “point” used as a verb)
   - filler adjectives (“nice”, “good”, “bad” unless explicitly meaningful)
   - full sentences
   - vague conceptual nouns (“something”, “stuff”, “content”, “information”)

4. Topic relevance filter:
   Extract terms **only if relevant to the topic’s definition** as provided below.
   If a keyword does not match the topic definition, do NOT extract it.

5. Formatting rules:
   - lowercase only
   - singular form for nouns and noun phrases
   - keep hyphenated or multiword adjectives (e.g., “hard to read”, “attention catching”)
   - U.S. English spelling (“color”, not “colour”)

6. Multiple keywords per phrase:
   Example:
     “cluttered and hard to read labels”
     → ["cluttered", "hard to read", "label"]

7. Output must be a valid JSON string of the form:
   {{
     "TopicA": ["keyword1", "keyword2"],
     "TopicB": ["keyword"],
     ...
   }}

   - No code blocks
   - No escape characters
   - Must be parsable via json.loads()

===========================================================
INPUT: Topics and Definitions
{topics_str}

INPUT: Extracted Phrases
{extracted}

===========================================================

Now produce the JSON output.
"""


def gpt_keyword_extract(text):

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "user",
                "content": text,
            },
        ],
    )

    try:
        content = response.choices[0].message.content
        return json.dumps(json.loads(content), indent=2)
    except:
        return "--NOT JSON--\n" + content

In [ ]:
def batch_keyterm_extract(idx, row, select):
    try:
        extracted = row[select]  # ExtractedPhrasesMoreComplex or ExtractedPhrasesLessComplex
        if pd.isna(extracted):
            return idx, None
        prompt = keyword_extract_prompt(extracted)
        stem = gpt_keyword_extract(prompt)
        return idx, stem
    except Exception as e:
        print(e)
        return idx, None

# set which comment to use
select = "ExtractedPhrasesLessComplex" # ExtractedPhrasesMoreComplex or ExtractedPhrasesLessComplex

results = []
with ThreadPoolExecutor() as executor:
    futures = {
        executor.submit(batch_keyterm_extract, idx, row, select): idx
        for idx, row in df.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            idx, stem = future.result()
            if stem is not None:
                results.append((idx, stem))
        except Exception as e:
            print(f'error: {idx}. {e}')

if select == "ExtractedPhrasesMoreComplex":
    result_col = "ExtractedKeywordsMoreComplex"
elif select == "ExtractedPhrasesLessComplex":
    result_col = "ExtractedKeywordsLessComplex"

for idx, stem in results:
    df.loc[idx, result_col] = stem

df[['index', result_col]].to_csv(f'data_processing/{result_col}.csv', index=False)
df.head()

## 4.1 Seperate Cognitive Load into new columns

In [ ]:
def extract_effect(val):
    """
    Extracts the 'Immediacy / Cognitive Load' list,
    returns a JSON string:
      {"Immediacy / Cognitive Load": [...]}
    If empty or missing, return "".
    """
    # If empty or not a string
    if not isinstance(val, str) or val.strip() == "":
        return ""

    # Attempt to load the JSON
    try:
        data = json.loads(val)
    except Exception:
        return ""

    # Extract the field
    items = data.get("Immediacy / Cognitive Load", [])

    # If no results, return empty string
    if not items:
        return ""

    # Otherwise return JSON string with key and items
    return json.dumps({"Immediacy / Cognitive Load": items})

# Apply to both columns
df["ExtractedEffectMoreComplex"] = df["ExtractedKeywordsMoreComplex"].apply(extract_effect)
df["ExtractedEffectLessComplex"] = df["ExtractedKeywordsLessComplex"].apply(extract_effect)

df[['index', 'ExtractedEffectMoreComplex', 'ExtractedEffectLessComplex']].to_csv('data_processing/ExtractedEffect.csv', index=False)
df[['index', 'ExtractedEffectMoreComplex', 'ExtractedEffectLessComplex']].head()


# 5. Keyword Curation

In [ ]:
# read in the latest main table "0.postquestionare_all"

spreadsheet_id = "1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc"
sheet_id = "1103808983"
df_original = load_sheet(spreadsheet_id, sheet_id)
print(f'Number of records is {len(df_original)}')

df = df_original
df = df[df['ToRemove']!=1]
print(f'Number of records after filtering is {len(df)}')
# df.columns

## 5.0 Keyword Tabulation with Count

In [ ]:
# do this for both moreComplex and lessComplex columns
fields = ["(to merge with next col) CuratePhrasesMoreComplex", "CurateCognitiveMoreComplex",
          "(to merge with next col) CuratePhrasesLessComplex", "CurateCognitiveLessComplex",]

df[fields].head()

In [ ]:
### Mrege into one column

def safe_load_json(s):
    """Safely load JSON string; return {} if invalid or empty."""
    if not isinstance(s, str) or s.strip() == "":
        return {}
    try:
        return json.loads(s)
    except:
        return {}

def merge_json_dicts(a, b):
    """
    Merge two dicts of the form:
      topic -> [keywords]
    Deduplicates keyword lists.
    """
    out = {}

    for d in (a, b):
        for topic, kws in d.items():
            if topic not in out:
                out[topic] = []
            if isinstance(kws, list):
                out[topic].extend([k for k in kws if isinstance(k, str)])

    # Deduplicate and sort
    for topic in out:
        out[topic] = sorted(set(out[topic]))

    return out


# ---- Column names ----
raw_more   = "(to merge with next col) CuratePhrasesMoreComplex"
cog_more   = "CurateCognitiveMoreComplex"

raw_less   = "(to merge with next col) CuratePhrasesLessComplex"
cog_less   = "CurateCognitiveLessComplex"


# ---- Merge into new fields ----

df_original["CuratePhrasesMoreComplex"] = df_original.apply(
    lambda row: (
        "" if len(
            merge_json_dicts(
                safe_load_json(row.get(raw_more)),
                safe_load_json(row.get(cog_more))
            )
        ) == 0 else json.dumps(
            merge_json_dicts(
                safe_load_json(row.get(raw_more)),
                safe_load_json(row.get(cog_more))
            ),
            indent=2
        )
    ),
    axis=1
)

df_original["CuratePhrasesLessComplex"] = df_original.apply(
    lambda row: (
        "" if len(
            merge_json_dicts(
                safe_load_json(row.get(raw_less)),
                safe_load_json(row.get(cog_less))
            )
        ) == 0 else json.dumps(
            merge_json_dicts(
                safe_load_json(row.get(raw_less)),
                safe_load_json(row.get(cog_less))
            ),
            indent=2
        )
    ),
    axis=1
)

print(f'Number of merged records is {len(df_original)}')

df_original[["index", "CuratePhrasesMoreComplex", "CuratePhrasesLessComplex"]].to_csv("data_processing/CuratedPhrasesCombined.csv", index=False)


In [ ]:
# build a dict with count and sentiment

kw_cnt = {}

cols_to_read = [
    ("CuratePhrasesMoreComplex", "+"),
    ("CuratePhrasesLessComplex", "-")
]

for idx, row in df_original.iterrows():
    if row.ToRemove == 1:
        continue

    for col, sentiment in cols_to_read:
        extracted = row.get(col, None)

        if pd.isna(extracted) or not isinstance(extracted, str) or extracted.strip() == "":
            continue

        try:
            obj = json.loads(extracted)
        except:
            continue

        # Count keywords
        for topic, keywords in obj.items():
            kw_cnt.setdefault(topic, {})

            for keyword in keywords:
                # Normalize
                keyword_clean = keyword.replace("colour", "color").strip()

                # Store as tuple (keyword, sentiment)
                key = (keyword_clean, sentiment)

                kw_cnt[topic][key] = kw_cnt[topic].get(key, 0) + 1

# Sort within each topic alphabetically by keyword
for topic in kw_cnt:
    kw_cnt[topic] = dict(sorted(kw_cnt[topic].items(), key=lambda x: x[0][0]))

# total number of (keyword, sentiment) combos
sum(len(v) for v in kw_cnt.values())

In [ ]:

# Prepare output table
topic_tables = []

for topic, kw_dict in kw_cnt.items():
    rows = []
    for (keyword, sentiment), count in kw_dict.items():
        rows.append({
            f"{topic}": keyword,
            f"{topic}_sentiment": sentiment,
            f"{topic}_count": count
        })

    # Convert to DF for this topic
    topic_df = pd.DataFrame(rows)

    # Sort keywords alphabetically (optional)
    topic_df = topic_df.sort_values(by=f"{topic}").reset_index(drop=True)

    topic_tables.append(topic_df)

# Now we align all topic tables by row count
max_rows = max(len(t) for t in topic_tables)

# Pad each topic block to max_rows
aligned_tables = []
for t in topic_tables:
    pad_rows = max_rows - len(t)
    if pad_rows > 0:
        # add empty rows to bottom
        t = pd.concat([t, pd.DataFrame([{}] * pad_rows)], axis=0, ignore_index=True)
    aligned_tables.append(t)

# Final wide table
kw_cnt_table = pd.concat(aligned_tables, axis=1)

# Save
kw_cnt_table.to_csv("data_processing/keyword_count_expanded.csv", index=False)

kw_cnt_table


### 5.0.A Code to backfill AI- and Human-curated terms into the above table
Used only to merge with prior curation results

In [ ]:
# read in the latest table "1.keyword.Z&AA(AI+humanCuration)"

spreadsheet_id = "1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc"
sheet_id = "1841871821"
df_kw_t1 = load_sheet(spreadsheet_id, sheet_id)
print(f'Number of records is {len(df_kw_t1)}')

print(f'Number of records after filtering is {len(df_kw_t1)}')
# df_kw_t1.columns

In [ ]:

def normalize_kw(s):
    """Normalize keyword for matching."""
    if not isinstance(s, str):
        return ""
    s = re.sub(r"^\(\d+\)\s*", "", s)   # remove "(n)" count prefix
    s = s.strip().lower()               # lowercase, strip
    s = s.rstrip(".")                   # remove trailing period
    s = re.sub(r"\s+", " ", s)          # normalize internal whitespace
    return s


# ------------------------------------------------------------
# 1. Build lookup tables from df_kw_t1
# ------------------------------------------------------------

lookup = {}  # topic → { normalized_keyword → (AIcurated, HumanCuration) }

for topic in topic_names:
    col_keyword = topic
    col_ai = f"{topic} (AIcurated)"
    col_human = f"{topic} (HumanCuration)"

    if col_keyword not in df_kw_t1.columns:
        continue

    topic_map = {}
    for _, row in df_kw_t1.iterrows():
        raw_key = row.get(col_keyword, "")
        key_norm = normalize_kw(raw_key)

        if key_norm != "":
            ai_val = row.get(col_ai, "")
            human_val = row.get(col_human, "")

            topic_map[key_norm] = (
                ai_val if isinstance(ai_val, str) else "",
                human_val if isinstance(human_val, str) else ""
            )

    lookup[topic] = topic_map


# ------------------------------------------------------------
# 2. Add two new columns per topic to kw_cnt_table
# ------------------------------------------------------------

for topic in topic_names:
    key_col = f"{topic}"
    ai_col  = f"{topic} (AIcurated)"
    hum_col = f"{topic} (HumanCuration)"
    sent_col = f"{topic}_sentiment"   # sentiment column already in kw_cnt_table

    # Initialize new columns if not present
    if ai_col not in kw_cnt_table.columns:
        kw_cnt_table[ai_col] = ""
    if hum_col not in kw_cnt_table.columns:
        kw_cnt_table[hum_col] = ""

    topic_lookup = lookup.get(topic, {})

    for i, kw in kw_cnt_table[key_col].items():
        if not isinstance(kw, str) or kw.strip() == "":
            continue

        key_norm = normalize_kw(kw)   # normalized key for lookup

        ai_val, hum_val = topic_lookup.get(key_norm, ("", ""))

        # sentiment already present, but wrap in parentheses
        sentiment_raw = kw_cnt_table.at[i, sent_col]
        if isinstance(sentiment_raw, str) and sentiment_raw.strip() in ["+", "-"]:
            kw_cnt_table.at[i, sent_col] = f"({sentiment_raw.strip()})"
        else:
            kw_cnt_table.at[i, sent_col] = ""

        # assign values
        kw_cnt_table.at[i, ai_col] = ai_val
        kw_cnt_table.at[i, hum_col] = hum_val


# ------------------------------------------------------------
# Done
# ------------------------------------------------------------

kw_cnt_table.to_csv("data_processing/keyword_count_with_ai_human.csv", index=False)
kw_cnt_table


### 5.0.B Generated a new table "1.keyword.S&Y(AI+humanCuration)"
Contains (+)/(-) and key phrase count

## 5.1 AI Curation Step
Use GPT-5.1 to curate the above raw phrases to generate column "topic (AIcurated)".

This step will consolidate raw phrases into less key phrases for the next step

## 5.2 Human Curation to Generate Final Keywords
Human curate the AI-curated column to further consolidate

In [ ]:
# open the final curated table

spreadsheet_id = "1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc"
sheet_kw_extract_id = "1166121277"
df_kw_final = load_sheet(spreadsheet_id, sheet_kw_extract_id)
print(f'Number of records is {len(df_kw_final)}')
df_kw_final.head()

## 5.3 Generate "A.keyword.Count" Table

In [ ]:
import os

topic_tables = []

for topic in topic_names:

    col_raw   = f"{topic}"                    # original keyword (not used for grouping)
    col_sent  = f"{topic}_sentiment"
    col_cnt   = f"{topic}_count"
    col_human = f"{topic} (HumanCuration)"    # <-- group by this one

    # Check columns exist
    if col_human not in df_kw_final.columns:
        continue

    # Extract relevant columns
    sub = df_kw_final[[col_human, col_sent, col_cnt]].copy()

    # Keep rows that have a human-curated keyword
    sub = sub[sub[col_human].astype(str).str.strip() != ""]

    # Normalize count
    sub[col_cnt] = pd.to_numeric(sub[col_cnt], errors="coerce").fillna(0)

    # ---- GROUP BY HUMAN CURATED KEYWORD + SENTIMENT ----
    grp = (
        sub.groupby([col_human, col_sent], dropna=False)[col_cnt]
           .sum()
           .reset_index()
    )

    # Sort
    grp = grp.sort_values(by=col_human).reset_index(drop=True)

    # Rename to final output format
    grp = grp.rename(columns={
        col_human: f"{topic}",
        col_sent: f"{topic}_sentiment",
        col_cnt: f"{topic}_total_count"
    })

    topic_tables.append(grp)

# -----------------------------------------------
# Align all topic blocks to same # rows
# -----------------------------------------------

max_rows = max(len(t) for t in topic_tables)

aligned_tables = []
for t in topic_tables:
    pad = max_rows - len(t)
    if pad > 0:
        blank = pd.DataFrame([{col: "" for col in t.columns}] * pad)
        t = pd.concat([t, blank], ignore_index=True)
    aligned_tables.append(t)

# Final wide table
df_kw_output = pd.concat(aligned_tables, axis=1)

df_kw_output.to_csv("data_processing/keyword_count_unique_by_human_curated.csv", index=False)
df_kw_output

## 5.4 Calculate Keyword Count for Each VisType
(Need to use "6.imageTypeTopics" that was generated following Section 5.5 and Section 6.1 and even Section 6.2 if VC scores are sorted)

In [ ]:
# load the latest "6.imageTypeTopics" table

spreadsheet_id = "1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc"
sheet_id = "1390591889"
df_images = load_sheet(spreadsheet_id, sheet_id)
print(f'Number of records is {len(df_images)}')
df_images.head()

In [ ]:
def expand_topic_keywords(cell):
    """
    Parse a keyword cell like:
      "(+) font; (-) explanation text; (+) more text"

    Returns a list of tuples:
      [("font", "+"), ("explanation text", "-"), ("more text", "+")]
    """
    if not isinstance(cell, str) or cell.strip() == "":
        return []

    parts = [p.strip() for p in cell.split(";") if p.strip()]
    cleaned = []

    for p in parts:
        # detect sentiment
        if p.startswith("(+)"):
            sentiment = "(+)"
            kw = p[3:].strip()
        elif p.startswith("(-)"):
            sentiment = "(-)"
            kw = p[3:].strip()
        else:
            sentiment = ""
            kw = p.strip()

        if kw:
            cleaned.append((kw, sentiment))

    return cleaned


def count_keywords_by_vistype(df_images, topic_names, vistype_list):
    """
    Build keyword × VisType count table, including sentiment.
    """

    rows = []

    for topic in topic_names:

        # Dict:
        #   (keyword, sentiment) -> set(images)
        kw_img = defaultdict(set)

        # Dict:
        #   (keyword, sentiment) -> VisType -> set(images)
        kw_vt = defaultdict(lambda: defaultdict(set))

        for _, row in df_images.iterrows():

            img = row["ImageName"]
            vt  = row["VisType"]
            cell = row[topic]

            kw_list = expand_topic_keywords(cell)

            # Deduplicate within each image
            for (kw, sentiment) in set(kw_list):
                key = (kw, sentiment)

                kw_img[key].add(img)

                if vt:
                    kw_vt[key][vt].add(img)

        # Build output rows
        for (kw, sentiment), imgset in kw_img.items():

            row_dict = {
                "topic": topic,
                "keyword": kw,
                "sentiment": sentiment,
                "num_images": len(imgset)
            }

            for vt in vistype_list:
                row_dict[vt] = len(kw_vt[(kw, sentiment)].get(vt, set()))

            rows.append(row_dict)

    return pd.DataFrame(rows)


# ---------------------------
# RUN THE ANALYSIS
# ---------------------------

df_keyword_counts = count_keywords_by_vistype(df_images, topic_names, VisTypes)
print(f'Number of records is {len(df_keyword_counts)}')

df_keyword_counts.to_csv("data_processing/keyword_counts_by_vistype.csv", index=False)
df_keyword_counts.head()

## 5.5 Get Final Keywords back to Each Image (Final.MappedBack)

In [ ]:
# open sheet "1.keyword.S&Y(AI+humanCuration)"
spreadsheet_id = "1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc"
sheet_kw_extract_id = "1166121277"
df_kw_curate_merge = load_sheet(spreadsheet_id, sheet_kw_extract_id)
print(f'Number of records is {len(df_kw_curate_merge)}')
df_kw_curate_merge.head(2)

In [ ]:
### Pipeline to create CuratedFinalKeywordsMoreComplex and CuratedFinalKeywordsLessComplex

def normalize_raw_phrase(s):
    """Lowercase, strip whitespace and remove trailing period."""
    if not isinstance(s, str):
        return ""
    s = s.lower().strip()
    ### s = re.sub(r"^\(\d+\)\s*", "", s)   # remove leading (n)
    s = s.rstrip(".")                  # remove trailing period
    return s.strip()


# 1. Raw -> Final (per topic) lookup
lookup = {t: {} for t in topic_names}

for topic in topic_names:
    raw_col = topic
    curated_col = f"{topic} (HumanCuration)"

    if raw_col not in df_kw_curate_merge.columns or curated_col not in df_kw_curate_merge.columns:
        print(f"Missing columns for topic: {topic}")
        continue

    for _, row in df_kw_curate_merge.iterrows():
        raw_val = row.get(raw_col, "")
        curated_val = row.get(curated_col, "")

        # Skip empty or removed entries
        if not isinstance(raw_val, str) or not isinstance(curated_val, str):
            continue
        if curated_val.strip() == "":
            continue

        # Normalize raw phrase
        norm_raw = normalize_raw_phrase(raw_val)
        if norm_raw == "":
            continue

        # Store mapping (string → string)
        lookup[topic][norm_raw] = curated_val.strip()


# ----------------------------------------
# 2. Extract raw phrases per topic from df
# ----------------------------------------

def extract_topic_raw_phrases(val):
    """Handle both dict and JSON string."""
    if isinstance(val, dict):
        return val
    if isinstance(val, str) and val.strip():
        try:
            return json.loads(val)
        except:
            return {}
    return {}


# ----------------------------------------
# 3. Convert raw phrases → final curated keywords (per topic)
# ----------------------------------------

def process_raw_phrase_json(raw_json):
    """
    Process ONE merged raw keyword JSON column (More or Less).
    raw_json is the JSON string for one image.
    """
    # Parse raw dict
    all_topics_raw = extract_topic_raw_phrases(raw_json)

    final_result = {}

    # Loop topics
    for topic in topic_names:
        raw_list = all_topics_raw.get(topic, [])
        if not raw_list:
            continue

        topic_final = []
        for raw_phrase in raw_list:
            norm = normalize_raw_phrase(raw_phrase)
            curated = lookup[topic].get(norm)
            if curated:
                topic_final.append(curated)

        if topic_final:
            final_result[topic] = sorted(set(topic_final))

    if not final_result:
        return ""

    return json.dumps(final_result, indent=2)


# ----------------------------------------
# 4. Apply to df
# ----------------------------------------

df_original["CuratedFinalKeywordsMoreComplex"] = df_original["CuratePhrasesMoreComplex"].apply(
    lambda x: process_raw_phrase_json(x)
)

df_original["CuratedFinalKeywordsLessComplex"] = df_original["CuratePhrasesLessComplex"].apply(
    lambda x: process_raw_phrase_json(x)
)

df_original[[
    'index',
    'CuratedFinalKeywordsMoreComplex',
    'CuratedFinalKeywordsLessComplex'
]].to_csv("data_processing/CuratedFinalKeywords_MoreAndLess.csv", index=False)

df_original[[
    'index',
    'CuratedFinalKeywordsMoreComplex',
    'CuratedFinalKeywordsLessComplex'
]]


# 6. Image Complexity

## 6.1 Fill in the 7 topic/metric columns for each image

In [ ]:
# load the latest main table
spreadsheet_id = "1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc"
sheet_id = "1103808983"
df_original = load_sheet(spreadsheet_id, sheet_id)
print(f'Number of records is {len(df_original)}')

df = df_original[df_original['ToRemove']!=1]
print(f'Number of records after filtering is {len(df)}')
df.columns


In [ ]:
# Merge final keywords

def merge_keyword_json_into_image(topic_map, json_str, valid_topics, prefix="", sentiment=False):
    """
    Merge JSON string (topics → [keywords]) directly into topic_map (topic -> set(prefixed_keywords)).
    Only keeps topics in valid_topics. Applies (+)/(-) prefix if sentiment=True.
    """
    if not isinstance(json_str, str) or json_str.strip() == "":
        return

    try:
        data = json.loads(json_str)
    except Exception:
        return

    for topic, keywords in data.items():
        if topic not in valid_topics:
            continue

        if not isinstance(keywords, list):
            continue

        for k in keywords:
            if not isinstance(k, str):
                continue
            kw = k.strip()
            if not kw:
                continue

            if sentiment and prefix:
                topic_map[topic].add(f"{prefix}{kw}")
            else:
                topic_map[topic].add(kw)


def assemble_keywords(df, topic_names, sentiment=False):
    """
    Produce ONE final row per unique imageName.

    For each row in df:
      - moreComplexImageName gets all its keywords as "(+) " (if sentiment=True)
      - lessComplexImageName gets all its keywords as "(-) " (if sentiment=True)

    If the same image appears in multiple rows or sometimes as more, sometimes as less:
      - all (+) and (-) keywords are combined per topic.
    """

    valid_topics = set(topic_names)

    # imageName -> {topic -> set(prefixed_keywords)}
    image_dict = defaultdict(lambda: defaultdict(set))

    # imageName -> VisType (first non-empty wins)
    vistype_dict = {}

    # Helper: ensure image exists and merge its keywords
    def add_keywords_for_image(img_name, js, prefix, vistype_value):
        if not isinstance(img_name, str):
            return
        img_name_clean = img_name.strip()
        if not img_name_clean:
            return

        # Ensure the image exists in image_dict even if it has no keywords yet
        _ = image_dict[img_name_clean]  # touch / create entry

        # Merge keywords directly into this image's topic map
        merge_keyword_json_into_image(
            topic_map=image_dict[img_name_clean],
            json_str=js,
            valid_topics=valid_topics,
            prefix=prefix,
            sentiment=sentiment
        )

        # Store VisType if not set yet
        if img_name_clean not in vistype_dict and isinstance(vistype_value, str):
            vt = vistype_value.strip()
            if vt:
                vistype_dict[img_name_clean] = vt

    # ======================================================
    # PASS 1 — loop through ALL rows
    # ======================================================
    for _, row in df.iterrows():
        # MORE COMPLEX side: "(+)"
        img_more = row.get("moreComplexImageName")
        js_more  = row.get("(Final.MappedBack)HumanCuratedFinalKeywordsMoreComplex")
        vt_more  = row.get("VisTypesMoreComplexImageType")

        add_keywords_for_image(
            img_name=img_more,
            js=js_more,
            prefix="(+) " if sentiment else "",
            vistype_value=vt_more,
        )

        # LESS COMPLEX side: "(-)"
        img_less = row.get("lessComplexImageName")
        js_less  = row.get("(Final.MappedBack)HumanCuratedFinalKeywordsLessComplex")
        vt_less  = row.get("VisTypesLessComplexImageType")

        add_keywords_for_image(
            img_name=img_less,
            js=js_less,
            prefix="(-) " if sentiment else "",
            vistype_value=vt_less,
        )

    # ======================================================
    # PASS 2 — build final rows: one row per unique image
    # ======================================================
    final_rows = []

    for img, topic_map in image_dict.items():
        # Deduplicate & sort after prefixing for stable output
        cleaned = {
            topic: sorted(list(kws))
            for topic, kws in topic_map.items()
            if len(kws) > 0
        }

        # Skip images with no keywords (cleaned == {})
        if len(cleaned) == 0:
            continue

        # NOTE: we no longer drop images with no keywords.
        final_rows.append({
            "imageName": img,
            "VisType": vistype_dict.get(img),
            "KeywordsJSON": json.dumps(cleaned, indent=2)
        })

    return final_rows


# Use it as before
merged = assemble_keywords(df, topic_names, sentiment=True)
df_out = pd.DataFrame(merged)
df_out.rename(columns={'imageName': 'ImageName'}, inplace=True)

df_out.to_csv('data_processing/image_all_keywords.csv', index=False)
print(f'Number of images is {len(df_out)}')
df_out.head()

In [ ]:
# now change the format of the table

def expand_merged_keywords(merged_list, topic_names):
    """
    Convert merged image keyword JSON (list of dicts) into
    a table with:
      - imageName
      - VisType
      - MergedKeywords (JSON string)
      - Topics (topic names joined by '; ')
      - One column per topic (keywords joined by '; ')
    """

    valid_topics = list(topic_names)
    rows = []

    for rec in merged_list:
        img = rec["imageName"]
        vistype = rec.get("VisType")
        json_str = rec["KeywordsJSON"]

        data = json.loads(json_str)   # dict: topic -> list(keywords)

        # Topic list
        topic_list = sorted(data.keys())
        topic_str = "; ".join(topic_list)

        # Prepare row
        row = {
            "imageName": img,
            "VisType": vistype,
            "MergedKeywords": json_str,
            "Topics": topic_str
        }

        # Add one column per topic
        for t in valid_topics:
            row[t] = "; ".join(data.get(t, []))

        rows.append(row)

    return pd.DataFrame(rows)


# generate expanded table
expanded_df = expand_merged_keywords(merged, topic_names)
expanded_df.rename(columns={'imageName': 'ImageName'}, inplace=True)

expanded_df.to_csv('data_processing/image_all_keywords_expanded.csv', index=False)
print(f'Number of images is {len(expanded_df)}')
expanded_df.head()


### Columns from the above are used to update sheet 6.imageTypeTopics (1390591889), which then can be used to generate sheet A.keyword.CountByVisTypes (1160817631)

## 6.2 Fill normalized VC score to the table

In [ ]:
# open sheet "6.imageTypeTopics"

spreadsheet_id = "1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc"
sheet_id = "1390591889"
image_type_vc = load_sheet(spreadsheet_id, sheet_id)
print(f'Number of records is {len(image_type_vc)}')
image_type_vc.head(2)

In [ ]:
# Get original S.VC to normalized value

spreadsheet_id = "1N6nK03QiPAM6AELb9kC3beETnKmG6u23"
sheet_id = "67202923"
vis_map = load_sheet(spreadsheet_id, sheet_id)

print(f'Number of images is {len(vis_map)}')

# ### normalize human_vc
# y = vis_map['S.VC']
# y = (y - min(y)) / (max(y) - min(y))
# y = (y + 0.0001) / 1.0001
# vis_map['NormalizedVC'] = y

# vis_map = vis_map[["Index", 'ImageName', 'S.VC', 'NormalizedVC']]
# vis_map.to_csv('original_vis_complexity_normalized.csv', index=False)

vis_map = vis_map[['ImageName', 'NormalizedVC']]
vis_map.head(2)

In [ ]:
# merge the two tables to get S.VC for each image
# df_image_vc = pd.merge(image_type_vc[['ImageName']], vis_map, on='ImageName', how='left')
df_image_vc = pd.merge(expanded_df, vis_map, on='ImageName', how='left')
df_image_vc.to_csv('data_processing/image_normalized_vc.csv', index=False)
print(f'Number of images is {len(df_image_vc)}')
df_image_vc.head()